# Cierre de la comparacion de pipelines

Ridge sirve como juez de los tratamientos. La eleccion del modelo final corresponde a la fase siguiente.

Actualización de selección (14/11/2026): el conjunto vigente tiene 15 predictoras al incorporar manualmente `es_fin_de_anio` (ADR 0015). Las salidas y métricas guardadas en esta notebook corresponden a las 14 variables originales; se conservan sin reejecución y deberán revisarse para la nueva selección.


## 1. Resultados de la comparacion anterior

Recuperamos los promedios de la ejecucion original para evitar repetir la grilla. Con el conjunto chico, elegimos la variante mas simple dentro de un desvio del minimo.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display
from sklearn.model_selection import TimeSeriesSplit

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent
sys.path.insert(0, str(raiz / "src"))
from auxiliar_functions import calcular_baselines, tabla_simple
from pipeline_sklearn import aprender_reglas_de_grupos, armar_pipeline, separar_xy
from preprocessing_pipeline import ejecutar_preprocesamiento

r = ejecutar_preprocesamiento(raiz / "data")
assert r["tabla_con_variables"].shape == (1212, 337)
train, val = r["entrenamiento"], r["validacion"]
columnas = r["reglas"]["variables"]["interacciones"]["variables_base"]

def mostrar(numero, titulo, tabla):
    print(f"Tabla {numero}. {titulo}")
    display(tabla_simple(tabla))

# Promedios y desvios registrados en la ejecucion original de la grilla.
resumen = pd.DataFrame({
    "Variante": list("ABCDEF"),
    "Tratamiento": ["Mediana y estandar", "Mediana y Yeo-Johnson estandarizado",
                    "Mediana con indicador y Yeo-Johnson estandarizado",
                    "Mediana, Yeo-Johnson y robusto", "Por grupos", "KNN y Yeo-Johnson"],
    "MAE chico": [3407.63, 3331.67, 3333.95, 3331.67, 3319.33, 3521.86],
    "Desvio chico": [289.82, 274.64, 256.71, 274.59, 242.96, 710.65],
    "MAE grande": [3886.67, 4257.92, 4810.10, 4378.50, 3645.81, 4107.66],
}).set_index("Variante")
mejor = resumen.loc[resumen["MAE chico"].idxmin()]
limite = mejor["MAE chico"] + mejor["Desvio chico"]
resumen["Dentro del limite"] = resumen["MAE chico"] <= limite
assert resumen.index[resumen["Dentro del limite"]][0] == "A"
mostrar(1, f"Comparacion anterior y limite de simplicidad ({limite:.2f} kW)", resumen)

Tabla 1. Comparacion anterior y limite de simplicidad (3562.29 kW)


,Tratamiento,MAE chico,Desvio chico,MAE grande,Dentro del limite
Variante,,,,,
A,Mediana y estandar,3407.63,289.82,3886.67,True
B,Mediana y Yeo-Johnson estandarizado,3331.67,274.64,4257.92,True
C,Mediana con indicador y Yeo-Johnson estandarizado,3333.95,256.71,4810.10,True
D,"Mediana, Yeo-Johnson y robusto",3331.67,274.59,4378.50,True
E,Por grupos,3319.33,242.96,3645.81,True
F,KNN y Yeo-Johnson,3521.86,710.65,4107.66,True


## 2. Errores por tramo y comprobacion de transformaciones

Cada pliegue tiene distinta cantidad de historia. Las fechas, el promedio y el desvio del consumo muestran que los bloques cubren periodos con niveles y variabilidad diferentes. El promedio de los pliegues resume varios periodos, mientras que la validacion externa mide un solo bloque. Las referencias y Ridge se miden sobre las mismas filas de cada bloque.

In [2]:
X, y = separar_xy(train.iloc[:-1], columnas)
X_val, y_val = separar_xy(val, columnas)
base_train = calcular_baselines(train)
base_val = calcular_baselines(pd.concat([train, val]))
filas = []
diferencias = []

def evaluar(nombre, ajuste, objetivo, entradas, real, baselines, fechas, variantes):
    base = baselines.loc[entradas.index].dropna()
    entradas, real = entradas.loc[base.index], real.loc[base.index]
    fila = {
        "Tramo": nombre, "Filas ajuste": len(ajuste), "Filas evaluacion": len(real),
        "Inicio": fechas.loc[real.index].min().date().isoformat(),
        "Fin": fechas.loc[real.index].max().date().isoformat(),
        "Consumo medio": real.mean(), "Desvio consumo": real.std(),
        "Persistencia": (real - base["persistencia"]).abs().mean(),
        "Media movil": (real - base["media_movil"]).abs().mean(),
    }
    grupos = aprender_reglas_de_grupos(ajuste, columnas)
    for variante, alpha in variantes.items():
        modelo = armar_pipeline(variante, columnas, grupos)
        modelo.set_params(regressor__alpha=alpha).fit(ajuste, objetivo)
        fila[variante] = (real - modelo.predict(entradas)).abs().mean()
        if variante == "D":
            pasos = modelo.named_steps["preprocessor"].named_transformers_["numericas"].named_steps
            assert not pasos["transformar"].standardize
            assert type(pasos["escalar"]).__name__ == "RobustScaler"
    return fila

for numero, (ajuste, evaluacion) in enumerate(TimeSeriesSplit(5).split(X), 1):
    if X.index[ajuste[-1]] + 1 == X.index[evaluacion[0]]:
        ajuste = ajuste[:-1]
    filas.append(evaluar(str(numero), X.iloc[ajuste], y.iloc[ajuste],
                         X.iloc[evaluacion], y.iloc[evaluacion], base_train, train["DIA"],
                         {"A": 100, "E": 0.01, "B": 0.01, "D": 0.01}))
externa = evaluar("Validacion", X, y, X_val, y_val, base_val, val["DIA"], {"A": 100})
pliegues = pd.DataFrame(filas).set_index("Tramo")
promedio = pliegues.mean(numeric_only=True)
contexto = pd.concat([pliegues, pd.DataFrame([externa]).set_index("Tramo")])
mostrar(2, "Fechas, consumo y MAE sobre filas comunes", contexto.drop(columns=["B", "D"]))
mostrar(3, "MAE por pliegue y promedio", pd.concat([
    pliegues[["Persistencia", "Media movil", "A", "E"]],
    promedio[["Persistencia", "Media movil", "A", "E"]].to_frame("Promedio").T,
]))
mostrar(4, "B y D por pliegue y promedio", pd.concat([
    pliegues[["B", "D"]], promedio[["B", "D"]].to_frame("Promedio").T,
]).assign(Diferencia=lambda t: t["D"] - t["B"]).round(6))

Tabla 2. Fechas, consumo y MAE sobre filas comunes


,Filas ajuste,Filas evaluacion,Inicio,Fin,Consumo medio,Desvio consumo,Persistencia,Media movil,A,E
Tramo,,,,,,,,,,
1,139,138,2020-11-22,2021-04-06,18322.14,6341.99,2576.87,2858.93,3915.03,3668.65
2,277,131,2021-04-07,2021-09-01,32349.17,4334.97,3478.95,3301.24,3241.62,3081.67
3,415,137,2021-09-02,2022-01-18,19340.39,8180.92,3240.04,3774.99,3266.43,3183.28
4,553,134,2022-01-19,2022-06-09,25234.47,6675.43,2976.93,3143.01,3233.42,3192.25
5,691,137,2022-06-10,2022-10-27,26195.53,6344.51,3974.88,3805.95,3381.64,3470.83
Validacion,830,172,2022-10-29,2023-04-27,17486.90,3902.69,2653.83,2489.91,2272.91,nan


Tabla 3. MAE por pliegue y promedio


,Persistencia,Media movil,A,E
1,2576.87,2858.93,3915.03,3668.65
2,3478.95,3301.24,3241.62,3081.67
3,3240.04,3774.99,3266.43,3183.28
4,2976.93,3143.01,3233.42,3192.25
5,3974.88,3805.95,3381.64,3470.83
Promedio,3249.53,3376.83,3407.63,3319.33


Tabla 4. B y D por pliegue y promedio


,B,D,Diferencia
1,3743.29,3743.25,-0.04
2,3094.41,3094.43,0.02
3,3194.24,3194.24,0.00
4,3145.26,3145.36,0.10
5,3481.18,3481.08,-0.10
Promedio,3331.67,3331.67,-0.00


## 3. Validacion externa y conclusion

A y E no muestran una mejora clara frente a ambas referencias en los pliegues. B y D usan escaladores distintos despues de la misma transformacion. Sus promedios redondeados coinciden, aunque los errores por pliegue son diferentes. La validacion externa informa cuanto reduce el error A frente a la mejor referencia, sin intervenir en la eleccion.

In [3]:
mejor_base = min(externa["Persistencia"], externa["Media movil"])
mejora = 100 * (mejor_base - externa["A"]) / mejor_base
mostrar(5, "Validacion externa e impacto sobre la mejor referencia", pd.DataFrame([{
    "Fechas": externa["Filas evaluacion"], "MAE A": externa["A"],
    "Persistencia": externa["Persistencia"], "Media movil": externa["Media movil"],
    "Mejora relativa (%)": mejora,
}]))

Tabla 5. Validacion externa e impacto sobre la mejor referencia


,Fechas,MAE A,Persistencia,Media movil,Mejora relativa (%)
0,172,2272.91,2653.83,2489.91,8.72
